In [1]:
import os

import json

import numpy as np

from UNLOADING_library.headers import hpc_headers
from Historia.shared.design_utils import read_labels

In [2]:
# Probably won't change

username                        = "crg17"
path2carputils                  = "/work/e348/e348/shared/carputils/"
carp_config_file                = "/work/e348/e348/shared/software/carpentry-system-petsc/carp.conf"
archer2_config_file             = None
path2unloading                  = f"/work/e348/e348/{username}/UNLOADING_library/"
env_folder                      = f"{path2unloading}/venv_UNLOADING_library/"
python_script_path_archer2      = f"{path2unloading}/run.py"

# Particular experiment

platform = "archer2"
folder_experiment_name          = "rodero_healthy/h11/scenarios/9/"
tags_setup_file_path_archer2    = f"/work/e348/e348/{username}/{folder_experiment_name}/json_files/tags.json"
general_setup_file_path_archer2 = f"/work/e348/e348/{username}/{folder_experiment_name}/json_files/settings_{platform}.json"

local_mesh_folder = f"/media/croderog/SeagateExpansionDrive/{folder_experiment_name}"
datafolder        = f"{local_mesh_folder}/data"
json_paramfolder  = f"{local_mesh_folder}/json_files"

In [3]:
### For the unloading the parameters affecting it are only in the X_mechanics

with open(f"{json_paramfolder}/settings_{platform}.json","r") as f:
    settings = json.load(f)
    
ncores = int(settings["nproc"])

X = np.loadtxt(f"{datafolder}/X_mechanics.txt",dtype=float)
N = X.shape[0]

In [4]:
xlabels_ = read_labels(f"{datafolder}/xlabels_mechanics.txt")

# The ones that are currently free in the run.py script in ARCHER2
input_params = ["a_ventricles",
                "bf_ventricles",
               "bfs_ventricles",
               "bt_ventricles",
               "a_atria",
               "bf_atria",
               "bfs_atria",
               "bt_atria",
               "k_peri",
               "EDP_lv",
               "EDP_rv",
               "a_lvrv",
               "bf_lv_scaling",
               "bf_rv_scaling",
               "bf_aa_scaling"]

for i in range(N):
	output_basename = f"unloading_{i}"
						 
	header = hpc_headers.write_archer2_header(jobname      = output_basename,
											  out_filename = f"{output_basename}.out",
											  walltime     = settings["walltime"],
											  ncores       = ncores)	

	env_variabiles = hpc_headers.write_env_variables(path2unloading      = path2unloading,
													 path2carputils      = path2carputils,
													 carp_config_file    = carp_config_file,
													 ncores              = ncores,
													 env_folder          = env_folder,
													 archer2_config_file = archer2_config_file)
	
	# read in parameters file
	parameters_file = f"{json_paramfolder}/{i}.json"
    
	with open(parameters_file,"r") as f:
		parameters = json.load(f)	


	runcommand  = ["python",python_script_path_archer2,"--platform","desktop"]
	runcommand += ["--overwrite-behaviour","overwrite"]
	runcommand += ["--np",str(ncores)]
	runcommand += ["--testname",output_basename]	

	runcommand += ["--tags_setup_file",tags_setup_file_path_archer2]
	runcommand += ["--general_setup_file",general_setup_file_path_archer2]	



	for ip in input_params:
		runcommand += [f"--{ip}",str(parameters["mechanics"][ip])]	

	runcommand = ' '.join(runcommand)	

	# -------------------------------------
	# write slrm file 
	slrm_script = f"{output_basename}.slrm"
	f = open(slrm_script,"w")	

	f.write(header)
	f.write(env_variabiles)
	f.write(runcommand)	

	f.close()

os.makedirs(f"{local_mesh_folder}/slrm", exist_ok=True)
os.system(f"mv ./unloading_*.slrm {local_mesh_folder}/slrm")

0